# Notebook 03 — MACE training (MacBook DEV / A100 smoke / A100 production)

This notebook is **training-only**.

It supports three practical profiles:

- **MacBook DEV** — CPU, debug subset, short run
- **A100 SMOKE** — CUDA, full dataset, very short run to verify GPU / memory / paths
- **A100 PROD** — CUDA, full dataset, real production baseline

Notes:
- `MODE = True` means **MacBook**
- `MODE = False` means **A100**
- On MacBook, use **CPU**, not `mps`, because `mace==0.3.14` can fail on Apple MPS due to float64 conversion during evaluation.


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


## Environment check


In [ ]:
import mace, torch, sys, subprocess
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("mps available:", torch.backends.mps.is_available())
print("cuda available:", torch.cuda.is_available())
print("mace version:", getattr(mace, "__version__", "unknown"))


## Training profile

Use the two switches below:

- `MODE = True`  → MacBook DEV
- `MODE = False` → A100

And for the A100:
- `SMOKE_TEST = True`  → short GPU verification run
- `SMOKE_TEST = False` → real production baseline


In [ ]:
from pathlib import Path

MODE = True          # True = MacBook, False = A100
SMOKE_TEST = False   # only relevant when MODE = False

if MODE:
    PROFILE = "MAC_DEV"
    DEVICE = "cpu"
    TRAIN_FILE = "../dataset/train_debug.extxyz"
    VALID_FILE = ""   # use random split from train
    VALID_FRACTION = 0.05

    R_MAX = 5.0
    MAX_ELL = 2
    NUM_INTERACTIONS = 2
    HIDDEN_IRREPS = "64x0e"
    NUM_RADIAL_BASIS = 8

    BATCH_SIZE = 1
    VALID_BATCH_SIZE = 1

    LR = 1e-3
    WEIGHT_DECAY = 1e-6
    MAX_EPOCHS = 20
else:
    DEVICE = "cuda"
    TRAIN_FILE = "../dataset/train.extxyz"   # change if your full dataset filename/path differs
    VALID_FILE = ""                          # optionally set an explicit valid file later
    VALID_FRACTION = 0.05

    R_MAX = 5.5
    MAX_ELL = 2
    NUM_INTERACTIONS = 3
    HIDDEN_IRREPS = "128x0e"
    NUM_RADIAL_BASIS = 8

    if SMOKE_TEST:
        PROFILE = "A100_SMOKE"
        BATCH_SIZE = 4
        VALID_BATCH_SIZE = 4
        LR = 5e-4
        WEIGHT_DECAY = 1e-6
        MAX_EPOCHS = 5
    else:
        PROFILE = "A100_PROD"
        BATCH_SIZE = 8
        VALID_BATCH_SIZE = 8
        LR = 5e-4
        WEIGHT_DECAY = 1e-6
        MAX_EPOCHS = 200

DTYPE = "float32"
FORCES_WEIGHT = 100
ENERGY_WEIGHT = 1
EMA_DECAY = 0.999

NAME = f"CuZr_{PROFILE}_r{R_MAX}"

print("PROFILE =", PROFILE)
print("NAME =", NAME)
print("DEVICE =", DEVICE)
print("TRAIN_FILE =", TRAIN_FILE)
print("VALID_FILE =", VALID_FILE if VALID_FILE else "<random split>")
print("R_MAX =", R_MAX)
print("HIDDEN_IRREPS =", HIDDEN_IRREPS)
print("NUM_INTERACTIONS =", NUM_INTERACTIONS)
print("BATCH_SIZE =", BATCH_SIZE)
print("LR =", LR)
print("MAX_EPOCHS =", MAX_EPOCHS)

p = Path(TRAIN_FILE)
print("train exists:", p.exists(), "size_MB:", round(p.stat().st_size/1e6, 2) if p.exists() else None)


## Optional dataset sanity check

This is useful when switching files or machines.


In [ ]:
from ase.io import read

atoms0 = read(TRAIN_FILE, index=0)
print("n_atoms in first frame:", len(atoms0))
print("info keys:", list(atoms0.info.keys()))
print("array keys:", list(atoms0.arrays.keys()))


## Build MACE training command

Current assumptions for the dataset:
- `energy_key = energy`
- `forces_key = forces`
- `E0s = average`

For future cleanup, you may later rewrite extxyz keys to `REF_energy` / `REF_forces`.


In [ ]:
import sys, shlex

cmd_parts = [
    sys.executable, "-m", "mace.cli.run_train",
    "--name", NAME,
    "--train_file", TRAIN_FILE,
    "--device", DEVICE,
    "--default_dtype", DTYPE,
    "--model", "MACE",
    "--r_max", str(R_MAX),
    "--apply_cutoff", "True",
    "--max_ell", str(MAX_ELL),
    "--num_interactions", str(NUM_INTERACTIONS),
    "--hidden_irreps", HIDDEN_IRREPS,
    "--radial_type", "bessel",
    "--num_radial_basis", str(NUM_RADIAL_BASIS),
    "--energy_key", "energy",
    "--forces_key", "forces",
    "--E0s", "average",
    "--loss", "ef",
    "--forces_weight", str(FORCES_WEIGHT),
    "--energy_weight", str(ENERGY_WEIGHT),
    "--optimizer", "adamw",
    "--lr", str(LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--ema",
    "--ema_decay", str(EMA_DECAY),
    "--batch_size", str(BATCH_SIZE),
    "--valid_batch_size", str(VALID_BATCH_SIZE),
    "--max_num_epochs", str(MAX_EPOCHS),
    "--eval_interval", "1",
]

if VALID_FILE:
    cmd_parts.extend(["--valid_file", VALID_FILE])
else:
    cmd_parts.extend(["--valid_fraction", str(VALID_FRACTION)])

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd_parts))


## Run training


In [ ]:
import subprocess, shlex

result = subprocess.run(
    cmd_parts,
    text=True,
    capture_output=True
)

print("RETURN CODE:", result.returncode)
print("\n===== STDOUT =====\n")
print(result.stdout)
print("\n===== STDERR =====\n")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("MACE training failed; see stdout/stderr above.")


## Find produced model artifacts


In [ ]:
from pathlib import Path

for p in sorted(Path(".").glob(f"**/{NAME}*")):
    print(p)


## Convert compiled model to LAMMPS-readable TorchScript

This uses the original MACE LAMMPS interface (`create_lammps_model`).
After running, look for a file ending in `-lammps.pt`.


In [ ]:
from pathlib import Path
import subprocess, sys

compiled_model = Path(f"{NAME}_compiled.model")
print("Compiled model exists:", compiled_model.exists(), compiled_model)

if compiled_model.exists():
    export_cmd = [sys.executable, "-m", "mace.cli.create_lammps_model", str(compiled_model)]
    print(" ".join(export_cmd))
    export_result = subprocess.run(export_cmd, text=True, capture_output=True)
    print("RETURN CODE:", export_result.returncode)
    print("\n===== STDOUT =====\n")
    print(export_result.stdout)
    print("\n===== STDERR =====\n")
    print(export_result.stderr)
    if export_result.returncode != 0:
        raise RuntimeError("LAMMPS export failed; see stdout/stderr above.")
else:
    print("Compiled model not found yet. Run training first.")


## Locate the final LAMMPS model file


In [ ]:
from pathlib import Path

for p in sorted(Path(".").glob("**/*lammps*.pt")):
    print(p)
